# Text Summarization con **PEGASUS** — Resumen abstractivo

> **Ejercicio en clase (NLP)** — *Text Summarization*. Implementamos **resumen abstractivo** con el modelo **PEGASUS** de Google (Hugging Face `transformers`).

A diferencia del resumen **extractivo** (que copia oraciones del texto original), PEGASUS es **abstractivo**: *genera* texto nuevo, parafraseando el contenido —igual que lo haria una persona.

**PEGASUS** (*Pre-training with Extracted Gap-sentences for Abstractive SUmmarization Sequence-to-sequence*) se pre-entrena enmascarando oraciones completas y pidiendole al modelo que las reconstruya, lo que lo hace especialmente bueno para resumir.

**Pipeline:**
1. Extraer un articulo de **Wikipedia** (mismo enfoque que el ejercicio G5).
2. Cargar el modelo PEGASUS pre-entrenado.
3. **Trocear** el texto (los articulos superan el limite de tokens del modelo).
4. Resumir cada trozo y combinar -> resumen final.
5. Evaluar la calidad con la metrica **ROUGE**.

**Herramientas:** `transformers` + `torch` (PEGASUS), `requests` (Wikipedia), `nltk` (segmentar oraciones), `evaluate` / `rouge-score` (metricas).

## 1. Instalacion e imports

> En **Google Colab** o un entorno nuevo, la siguiente celda instala todo. PEGASUS se descarga la primera vez (~2.3 GB para `pegasus-cnn_dailymail`). **Se recomienda GPU** (en Colab: *Entorno de ejecucion -> Cambiar tipo de entorno -> GPU*), aunque tambien corre en CPU (mas lento).

In [ ]:
# Instalacion robusta (Colab / entorno nuevo). Si ya estan, no hace nada.
%pip install -q transformers torch sentencepiece requests nltk evaluate rouge-score absl-py

import re
import textwrap
import requests
import torch
import nltk
from transformers import PegasusTokenizer, PegasusForConditionalGeneration

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__, '| Dispositivo:', DEVICE.upper())

## 2. Extraccion del texto desde Wikipedia

Reutilizamos la **API oficial de MediaWiki** (como en el ejercicio G5) para obtener el texto plano de un articulo en **ingles** (PEGASUS `cnn_dailymail` esta entrenado en ingles).

> Wikipedia exige una cabecera **User-Agent**; sin ella devuelve *403 Forbidden*.

In [ ]:
HEADERS = {'User-Agent': 'TextSummarizationExercise/1.0 (academico; contacto: estudiante@example.com)'}

def extraer_wikipedia(titulo, idioma='en'):
    '''Devuelve el texto plano de un articulo de Wikipedia via la API de MediaWiki.'''
    url = f'https://{idioma}.wikipedia.org/w/api.php'
    params = {
        'format': 'json', 'action': 'query', 'prop': 'extracts',
        'explaintext': 1, 'redirects': 1, 'titles': titulo,
    }
    r = requests.get(url, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    pagina = next(iter(r.json()['query']['pages'].values()))
    return pagina['title'], pagina.get('extract', '')

TITULO = 'Natural language processing'
titulo, texto = extraer_wikipedia(TITULO)

# Limpieza minima: quitamos encabezados de seccion (== Historia ==) y espacios sobrantes
texto = re.sub(r'==+\s*[^=]+\s*==+', ' ', texto)
texto = re.sub(r'\s+', ' ', texto).strip()

print(f'Articulo: {titulo} ({len(texto):,} caracteres, ~{len(texto.split()):,} palabras)\n')
print(texto[:600], '...')

## 3. Carga del modelo PEGASUS

Usamos **`google/pegasus-cnn_dailymail`**, afinado sobre noticias (CNN/Daily Mail), que produce resumenes de **varias oraciones** —apropiado para un articulo enciclopedico.

> **Alternativas:**
> - `google/pegasus-xsum` -> resumenes de **una sola oracion**, muy concisos.
> - `google/pegasus-large` -> modelo base generico.

El **tokenizer** convierte el texto a *tokens* y el modelo `...ForConditionalGeneration` genera el resumen (*seq2seq*).

In [ ]:
MODELO = 'google/pegasus-cnn_dailymail'

print(f'Cargando {MODELO} ... (la primera vez descarga el modelo)')
tokenizer = PegasusTokenizer.from_pretrained(MODELO)
model = PegasusForConditionalGeneration.from_pretrained(MODELO).to(DEVICE)

MAX_INPUT_TOKENS = tokenizer.model_max_length  # limite de tokens de entrada del modelo
print(f'Modelo listo en {DEVICE.upper()}. Limite de entrada: {MAX_INPUT_TOKENS} tokens.')

## 4. Troceo (*chunking*) del texto

Los modelos *transformer* tienen un **limite de tokens** de entrada (1024 para `pegasus-cnn_dailymail`). Un articulo de Wikipedia lo supera ampliamente, asi que lo dividimos en **trozos por oraciones** sin exceder ese limite. Cortar por oracion (no a mitad de palabra) preserva el sentido.

In [ ]:
def trocear_por_tokens(texto, tokenizer, max_tokens):
    '''Agrupa oraciones en trozos que no superen max_tokens (margen de seguridad incluido).'''
    limite = max_tokens - 30  # margen para tokens especiales
    oraciones = nltk.sent_tokenize(texto)
    trozos, actual, n_actual = [], [], 0
    for oracion in oraciones:
        n = len(tokenizer.tokenize(oracion))
        if n_actual + n > limite and actual:
            trozos.append(' '.join(actual))
            actual, n_actual = [], 0
        actual.append(oracion)
        n_actual += n
    if actual:
        trozos.append(' '.join(actual))
    return trozos

trozos = trocear_por_tokens(texto, tokenizer, MAX_INPUT_TOKENS)
print(f'El texto se dividio en {len(trozos)} trozos.')
for i, t in enumerate(trozos):
    print(f'  Trozo {i+1}: {len(tokenizer.tokenize(t))} tokens, {len(t.split())} palabras')

## 5. Generacion del resumen

Para cada trozo llamamos a `model.generate()`. Parametros clave de la generacion:

- **`num_beams`** — *beam search*: explora varias hipotesis y elige la mejor (mas calidad que *greedy*).
- **`max_length` / `min_length`** — longitud del resumen en tokens.
- **`length_penalty`** — > 1 favorece resumenes mas largos.
- **`no_repeat_ngram_size`** — evita repetir n-gramas (menos texto redundante).

In [ ]:
def resumir(texto, max_length=128, min_length=32, num_beams=5):
    '''Genera el resumen abstractivo de un fragmento con PEGASUS.'''
    inputs = tokenizer(texto, truncation=True, padding='longest',
                       max_length=MAX_INPUT_TOKENS, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_length=max_length, min_length=min_length,
            num_beams=num_beams, length_penalty=1.0,
            no_repeat_ngram_size=3, early_stopping=True,
        )
    resumen = tokenizer.decode(ids[0], skip_special_tokens=True)
    # pegasus-cnn_dailymail usa <n> como salto de linea entre oraciones
    return resumen.replace('<n>', ' ').strip()

# Resumimos cada trozo
resumenes_trozo = []
for i, trozo in enumerate(trozos):
    r = resumir(trozo)
    resumenes_trozo.append(r)
    print(f'--- Resumen del trozo {i+1} ---')
    print(textwrap.fill(r, width=100), '\n')

### 5.1 Resumen final

Concatenamos los resumenes parciales. Si el resultado sigue siendo largo, aplicamos un **segundo paso** de resumen (*resumen jerarquico*) para obtener una version final compacta.

In [ ]:
resumen_combinado = ' '.join(resumenes_trozo)

# Resumen jerarquico: si la combinacion es larga, la volvemos a resumir
if len(tokenizer.tokenize(resumen_combinado)) > MAX_INPUT_TOKENS - 30:
    print('El resumen combinado es largo -> aplicando segundo paso de resumen...\n')
    resumen_final = resumir(resumen_combinado, max_length=160, min_length=48)
else:
    resumen_final = resumen_combinado

print(f'RESUMEN FINAL de "{titulo}":\n')
print(textwrap.fill(resumen_final, width=100))

reduccion = 100 * (1 - len(resumen_final.split()) / len(texto.split()))
print(f'\nOriginal: {len(texto.split()):,} palabras  ->  Resumen: {len(resumen_final.split()):,} palabras')
print(f'Compresion: {reduccion:.1f}% mas corto.')

## 6. Evaluacion con ROUGE

**ROUGE** (*Recall-Oriented Understudy for Gisting Evaluation*) mide el solapamiento de n-gramas entre el resumen generado y un texto de **referencia**:

- **ROUGE-1 / ROUGE-2** -> solapamiento de *unigramas* / *bigramas*.
- **ROUGE-L** -> subsecuencia comun mas larga (captura fluidez/orden).

Como referencia usamos el **lead de la propia Wikipedia** —sus primeras oraciones, escritas por humanos para resumir el articulo. No es una referencia "oro" perfecta, pero sirve para tener una medida cuantitativa.

In [ ]:
import evaluate

# Referencia: primeras oraciones del articulo (resumen humano del lead de Wikipedia)
referencia = ' '.join(nltk.sent_tokenize(texto)[:5])

rouge = evaluate.load('rouge')
scores = rouge.compute(predictions=[resumen_final], references=[referencia])

print('Puntuaciones ROUGE (resumen PEGASUS vs. lead de Wikipedia):\n')
for metrica, valor in scores.items():
    print(f'  {metrica:10}: {valor:.4f}')

## 7. Conclusion

Implementamos **resumen abstractivo** de un articulo de Wikipedia con **PEGASUS**:

| Paso | Herramienta | Que hicimos |
|---|---|---|
| **Extraccion** | `requests` + API de Wikipedia | Articulo en ingles a texto plano |
| **Modelo** | `transformers` (PEGASUS) | `google/pegasus-cnn_dailymail` pre-entrenado |
| **Troceo** | `nltk` + tokenizer | Division por oraciones bajo el limite de tokens |
| **Generacion** | `model.generate` (*beam search*) | Resumen de cada trozo + resumen jerarquico |
| **Evaluacion** | `evaluate` (ROUGE) | Medida cuantitativa de calidad |

**Observaciones:**
- PEGASUS es **abstractivo**: parafrasea y genera texto nuevo, a diferencia de los metodos extractivos (TF-IDF, TextRank) que copian oraciones.
- El **limite de tokens** obliga a trocear documentos largos; el **resumen jerarquico** (resumir los resumenes) permite condensar articulos extensos.
- **ROUGE** da una metrica objetiva, pero un buen resumen abstractivo puede usar palabras distintas a la referencia y aun asi ser correcto —por eso conviene complementar con lectura humana.

> **Para experimentar:** cambia `MODELO` a `google/pegasus-xsum` para ver resumenes de una sola oracion, o prueba otro `TITULO` de Wikipedia.